# CBAM (Convolutional Block Attention Module)

CBAM, özellik haritasına sırayla **Channel Attention** ve **Spatial Attention** uygular.

- Channel Attention: Hangi kanalların önemli olduğunu öğrenir.
- Spatial Attention: Görüntünün hangi konumlarının önemli olduğunu öğrenir.


## 1. Kütüphaneler


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## 2. Channel Attention

Average Pooling ve Max Pooling sonuçları aynı MLP'den geçirilir. Sonuçlar toplanıp sigmoid ile 0–1 aralığına getirilir.


In [2]:
class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden_channels = max(channels // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        attention = self.sigmoid(avg_out + max_out)
        return x * attention


## 3. Spatial Attention

Kanal boyutunda ortalama ve maksimum alınır. İki harita birleştirilir ve 7×7 konvolüsyon ile konumsal dikkat haritası üretilir.


In [3]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        if kernel_size not in (3, 7):
            raise ValueError("kernel_size 3 veya 7 olmalı")

        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out = torch.max(x, dim=1, keepdim=True)[0]
        combined = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv(combined))
        return x * attention


## 4. CBAM modülü

İşlem sırası: önce Channel Attention, ardından Spatial Attention.


In [4]:
class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, spatial_kernel: int = 7):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x


## 5. Şekilleri test edelim

Örnek giriş: `[B, C, H, W] = [8, 64, 56, 56]`. CBAM, kanal ve uzamsal boyutları değiştirmez.


In [5]:
x = torch.randn(8, 64, 56, 56)
cbam = CBAM(channels=64)

y = cbam(x)

print("Input shape: ", x.shape)
print("Output shape:", y.shape)
assert y.shape == x.shape


Input shape:  torch.Size([8, 64, 56, 56])
Output shape: torch.Size([8, 64, 56, 56])


## 6. Bir CNN bloğuna CBAM ekleme örneği

CBAM'i konvolüsyon bloğunun çıkışına yerleştirebiliriz. Burada örnek bir blok gösterilmiştir.


In [6]:
class ConvBlockWithCBAM(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3,
                      stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.cbam = CBAM(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.cbam(x)
        return x

block = ConvBlockWithCBAM(3, 64, stride=2)
sample = torch.randn(4, 3, 224, 224)
result = block(sample)
print(result.shape)  # torch.Size([4, 64, 112, 112])


torch.Size([4, 64, 112, 112])


## Özet

- **Channel Attention:** `[B, C, 1, 1]` ağırlıkları üretir.
- **Spatial Attention:** `[B, 1, H, W]` ağırlıkları üretir.
- Her iki dikkat haritası da çarpma yoluyla giriş özelliklerine uygulanır.
- CBAM girişin `[B, C, H, W]` şeklini değiştirmez.
